## Bước 1 — Gắn Google Drive và kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/CARTE_ketqua"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_CSV = f"{SAVE_DIR}/ket_qua_cua_toi.csv"
print("Thư mục lưu kết quả:", SAVE_DIR)

!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -h | head -2

## Bước 2 — Cài thư viện

Cài `carte-ai` từ PyPI. Mô hình đã tiền huấn luyện `kg_pretrained.pt` (40 MB) đi kèm sẵn trong gói, không phải tải riêng.

Nếu Colab báo cần khởi động lại thì bấm **Khởi động lại phiên**, sau đó chạy tiếp từ Bước 3.

In [ ]:
!pip install carte-ai huggingface_hub -q
print("Đã cài xong.")

## Bước 3 — Tải mô hình ngôn ngữ FastText (~7 GB)

In [ ]:
import os, torch
from huggingface_hub import hf_hub_download
from carte_ai.configs.directory import config_directory

SAVE_DIR = "/content/drive/MyDrive/CARTE_ketqua"
OUT_CSV  = f"{SAVE_DIR}/ket_qua_cua_toi.csv"

FASTTEXT_PATH = hf_hub_download(repo_id="hi-paris/fastText", filename="cc.en.300.bin")

print("FastText          :", FASTTEXT_PATH)
print("CUDA khả dụng     :", torch.cuda.is_available())
p = config_directory["pretrained_model"]
print("Mô hình tiền h.l. :", round(os.path.getsize(p)/1e6, 1), "MB")

## Bước 4 — Cấu hình thực nghiệm

In [ ]:
TRAIN_SIZES = [32, 64]     # cỡ tập huấn luyện
SEEDS       = [1, 2]       # số lần lặp
NUM_MODEL   = 5            # số mô hình bagging (bài báo dùng 10)
DEVICE      = "cuda"
CHUNK       = 1500         # số dòng kiểm thử xử lý mỗi lô

print(f"Tổng số lượt chạy: 3 tập × {len(TRAIN_SIZES)} cỡ × {len(SEEDS)} seed "
      f"= {3*len(TRAIN_SIZES)*len(SEEDS)}")

## Bước 5 — Chuẩn bị bộ tiền xử lý và hàm chạy

Hai điểm kỹ thuật trong cell này:

1. `Table2GraphTransformer` được tạo **một lần duy nhất**. Đây là nơi nạp FastText vào bộ nhớ (~7 GB); tạo lại trong vòng lặp sẽ vượt giới hạn RAM của Colab.
2. Tập kiểm thử được chuyển thành đồ thị và dự đoán **theo từng lô**. Kết quả giống hệt khi xử lý một lần, nhưng bộ nhớ không phình theo kích thước tập kiểm thử.

In [ ]:
import time, gc, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore", category=FutureWarning)

# Giữ nguyên kiểu dữ liệu của cột khi cắt lô
try:
    pd.set_option("future.no_silent_downcasting", True)
except Exception:
    pass

from carte_ai import Table2GraphTransformer, CARTERegressor
from carte_ai.data.load_data import wina_pl, wine_dot_com_prices, wine_vivino_price

DATASETS = {
    "wina_pl":             {"loader": wina_pl,             "task": "regression"},
    "wine_dot_com_prices": {"loader": wine_dot_com_prices, "task": "regression"},
    "wine_vivino_price":   {"loader": wine_vivino_price,   "task": "regression"},
}

preprocessor = Table2GraphTransformer(fasttext_model_path=FASTTEXT_PATH)
print("Table2GraphTransformer sẵn sàng.")


def _predict_chunked(est, X_te):
    """Chuyển bảng thành đồ thị và dự đoán theo lô để giới hạn bộ nhớ."""
    obj_cols = [c for c in X_te.columns if X_te[c].dtype == object]
    out, n = [], len(X_te)
    for i in range(0, n, CHUNK):
        Xc = X_te.iloc[i:i + CHUNK].copy()
        for c in obj_cols:
            Xc[c] = Xc[c].astype(object)
        Xc_g = preprocessor.transform(Xc)
        out.append(np.asarray(est.predict(Xc_g)).ravel())
        del Xc, Xc_g
        gc.collect()
    return np.concatenate(out)


def run_one(data_name, num_train, seed):
    """Chạy một lượt: 1 tập dữ liệu × 1 cỡ huấn luyện × 1 seed."""
    cfg = DATASETS[data_name]

    X_tr, X_te, y_tr, y_te = cfg["loader"](num_train=num_train, random_state=seed)
    X_tr_g = preprocessor.fit_transform(X_tr, y=y_tr)

    est = CARTERegressor(num_model=NUM_MODEL, random_state=0, device=DEVICE,
                         n_jobs=1, disable_pbar=True)

    t0 = time.time()
    est.fit(X=X_tr_g, y=y_tr)
    score = r2_score(y_te, _predict_chunked(est, X_te))
    elapsed = time.time() - t0

    del X_tr_g, est
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"data_name": data_name, "num_train": num_train, "random_state": seed,
            "task": cfg["task"], "score": float(score), "giay": round(elapsed, 1),
            "n_train_rows": len(X_tr), "n_test_rows": len(X_te)}

print("Đã nạp hàm run_one.")

## Bước 6 — Chạy thực nghiệm

Mỗi lượt hoàn tất được ghi ngay vào Google Drive. Nếu phiên Colab bị ngắt, chạy lại cell này sẽ tự bỏ qua các lượt đã xong.

In [ ]:
done = set()
if os.path.exists(OUT_CSV):
    _d = pd.read_csv(OUT_CSV)
    done = set(zip(_d.data_name, _d.num_train, _d.random_state))
    print(f"Đã có {len(done)} lượt trong Drive, sẽ bỏ qua.\n")

todo = [(d, n, s) for d in DATASETS for n in TRAIN_SIZES for s in SEEDS
        if (d, n, s) not in done]
print(f"Còn {len(todo)} lượt cần chạy.\n")

for i, (d, n, s) in enumerate(todo, 1):
    print(f"[{i}/{len(todo)}] {d} | num_train={n} | seed={s} ...", flush=True)
    res = run_one(d, n, s)
    pd.DataFrame([res]).to_csv(OUT_CSV, mode="a",
                               header=not os.path.exists(OUT_CSV), index=False)
    print(f"      R² = {res['score']:.4f}   ({res['giay']:.0f} giây)\n")

print("Hoàn tất. Tệp kết quả:", OUT_CSV)
mine = pd.read_csv(OUT_CSV)
mine

## Bước 7 — Đối chiếu với kết quả công bố

Tệp `results_carte_baseline_singletable.csv` nằm trong chính kho mã nguồn của nhóm tác giả, chứa 78.980 lượt chạy của toàn bộ benchmark trong bài báo.

In [ ]:
REF_URL = ("https://raw.githubusercontent.com/soda-inria/carte/main/"
           "carte_ai/results/compiled_results/results_carte_baseline_singletable.csv")
ref_all = pd.read_csv(REF_URL)

print("Kết quả gốc của bài báo:")
print("  tổng số lượt :", len(ref_all))
print("  tập dữ liệu  :", ref_all.data_name.nunique())
print("  phương pháp  :", ref_all.model.nunique())
print("  cỡ train     :", sorted(ref_all.num_train.unique()))
print("  seed         :", sorted(ref_all.random_state.unique()))

ref = ref_all[ref_all.model == "carte-gnn"][
    ["data_name", "num_train", "random_state", "score"]].rename(columns={"score": "paper"})

cmp = (mine[["data_name", "num_train", "random_state", "score", "giay", "n_test_rows"]]
       .rename(columns={"score": "cua_toi"})
       .merge(ref, on=["data_name", "num_train", "random_state"], how="inner"))
cmp["chenh_lech"] = (cmp.cua_toi - cmp.paper).abs()
cmp = cmp.sort_values(["data_name", "num_train", "random_state"]).reset_index(drop=True)

print("\nBẢNG 1 — SO SÁNH TỪNG LƯỢT CHẠY")
display(cmp.round(4))

### Bảng tổng hợp và các chỉ số đánh giá mức độ tái lập

In [ ]:
summary = (cmp.groupby(["data_name", "num_train"])
              .agg(toi_TB=("cua_toi", "mean"), toi_DLC=("cua_toi", "std"),
                   paper_TB=("paper", "mean"), paper_DLC=("paper", "std"),
                   chenh_TB=("chenh_lech", "mean"), so_seed=("cua_toi", "count"))
              .round(4).reset_index())

print("BẢNG 2 — TỔNG HỢP THEO CẤU HÌNH")
display(summary)

r = np.corrcoef(cmp.cua_toi, cmp.paper)[0, 1]
print("\nCÁC CHỈ SỐ ĐÁNH GIÁ MỨC ĐỘ TÁI LẬP")
print(f"  Hệ số tương quan              : {r:.3f}")
print(f"  Chênh lệch tuyệt đối trung bình: {cmp.chenh_lech.mean():.4f}")
print(f"  Chênh lệch nhỏ nhất            : {cmp.chenh_lech.min():.4f}")
print(f"  Chênh lệch lớn nhất            : {cmp.chenh_lech.max():.4f}")
print(f"  Số lượt chênh dưới 0,05        : {(cmp.chenh_lech < 0.05).sum()}/{len(cmp)}")
print(f"  Số lượt chênh dưới 0,10        : {(cmp.chenh_lech < 0.10).sum()}/{len(cmp)}")
print(f"  Tổng thời gian tính toán       : {cmp.giay.sum():.0f} giây")

# Doi chieu voi khoang dao dong giua cac seed cua bai bao
print("\nĐỐI CHIẾU VỚI KHOẢNG ±1 ĐỘ LỆCH CHUẨN CỦA BÀI BÁO (10 seed)")
for _, row in summary.iterrows():
    s = ref_all[(ref_all.model == "carte-gnn")
              & (ref_all.data_name == row.data_name)
              & (ref_all.num_train == row.num_train)].score
    lo, hi = s.mean() - s.std(), s.mean() + s.std()
    trong = "nằm trong" if lo <= row.toi_TB <= hi else "nằm ngoài"
    print(f"  {row.data_name:22s} n={int(row.num_train):<4d} "
          f"bài báo {s.mean():.4f} ± {s.std():.4f}  "
          f"[{lo:.3f}; {hi:.3f}]   nhóm {row.toi_TB:.4f}  -> {trong}")

summary.to_csv(f"{SAVE_DIR}/bang_so_sanh.csv", index=False)

## Bước 8 — Biểu đồ

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

C_MINE, C_PAPER = "#2a78d6", "#eb6834"
INK, INK2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3df"
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "text.color": INK, "xtick.color": INK2, "ytick.color": INK2,
    "font.size": 10, "axes.grid": True, "grid.color": GRID, "grid.linewidth": .8,
    "axes.spines.top": False, "axes.spines.right": False})

# Hinh 1 - duong cong hoc
names = sorted(cmp.data_name.unique())
fig, axes = plt.subplots(1, len(names), figsize=(4.4*len(names), 4.1), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, names):
    g = (cmp[cmp.data_name == name].groupby("num_train")
         .agg(m=("cua_toi", "mean"), ms=("cua_toi", "std"),
              p=("paper", "mean"),   ps=("paper", "std")).reset_index())
    x = g.num_train.values
    ax.fill_between(x, g.p-g.ps, g.p+g.ps, color=C_PAPER, alpha=.13, lw=0)
    ax.fill_between(x, g.m-g.ms, g.m+g.ms, color=C_MINE,  alpha=.13, lw=0)
    ax.plot(x, g.p, color=C_PAPER, lw=2, marker="s", ms=8, mec=SURFACE, mew=2,
            label="Kết quả công bố")
    ax.plot(x, g.m, color=C_MINE,  lw=2, marker="o", ms=9, mec=SURFACE, mew=2,
            label="Nhóm chạy lại")
    ax.set_title(name, color=INK, fontsize=11, fontweight="bold", loc="left")
    ax.set_xlabel("Số mẫu huấn luyện"); ax.set_ylabel("R²")
    ax.set_xticks(x); ax.set_xticklabels([str(v) for v in x])
    ax.set_xlim(x.min()-6, x.max()+6)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(.5, 1.09))
plt.savefig(f"{SAVE_DIR}/hinh1_duongcong.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
plt.show()

# Hinh 2 - tuong quan
fig, ax = plt.subplots(figsize=(5.4, 5.2), constrained_layout=True)
lim = [0, 0.72]
ax.plot(lim, lim, color=INK2, lw=1.2, ls="--", zorder=1)
ax.scatter(cmp.paper, cmp.cua_toi, s=110, color=C_MINE,
           edgecolor=SURFACE, linewidth=2, zorder=3)
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
ax.set_xlabel("R² công bố trong bài báo"); ax.set_ylabel("R² nhóm chạy lại")
ax.set_title(f"Tương quan {len(cmp)} lượt chạy   r = {r:.3f}",
             color=INK, fontsize=12, fontweight="bold", loc="left")
ax.text(.04, .94, "Đường đứt nét: trùng khớp hoàn toàn",
        transform=ax.transAxes, color=INK2, fontsize=9)
plt.savefig(f"{SAVE_DIR}/hinh2_tuongquan.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
plt.show()

print("Đã lưu biểu đồ vào:", SAVE_DIR)